In [1]:
### Importing the required library 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
from sklearn.linear_model import Ridge
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')
from rdkit.Chem import MolStandardize
import joblib

import matplotlib.pyplot as plt
from rdkit import Chem
from rdkit import DataStructs
from rdkit.Chem import AllChem
from rdkit.Chem import Descriptors,rdMolDescriptors
#import openpyxl


# Environment - env

### Installation of the basic library 
from rdkit import Chem,DataStructs
from rdkit.Chem import AllChem
from rdkit.Chem.Draw import IPythonConsole
from rdkit.Chem import Descriptors
from rdkit.Chem import Lipinski
from rdkit.Chem import Crippen
from rdkit.Chem import Draw
import xgboost as xgboost
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score


## Binary ECFP (presence/absence)

When radiaus is larger than 1, then it is called ECFP. othervise it is called morgan fingerprint

Binary ECFP - give only presence, not how many times it appears.

In [2]:
def morgan_binary(smiles, radius=2, nBits=2048, use_chirality=True, use_features=False):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None: return None
    fp = AllChem.GetMorganFingerprintAsBitVect(
        mol, radius, nBits=nBits,
        useChirality=use_chirality, useFeatures=use_features
    )
    arr = np.zeros((nBits,), dtype=np.int8)
    Chem.DataStructs.ConvertToNumpyArray(fp, arr)
    return arr

## Count ECFP (recommended for regression)

Same hashed index space as binary ECFP, but each active position stores a count (frequency).

This is best for regression

In [3]:
def morgan_counts(smiles, radius=2, nBits=2048, use_chirality=True, use_features=False):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None: return None
    # Sparse map: {bit_index: count}
    fp = AllChem.GetHashedMorganFingerprint(
        mol, radius, nBits=nBits,
        useChirality=use_chirality, useFeatures=use_features
    )
    vec = np.zeros((nBits,), dtype=np.float32)
    for idx, cnt in fp.GetNonzeroElements().items():
        vec[idx] = float(cnt)
    return vec

## Interpreting bits (which substructure set the bit?)

Same fingerprint (binary or count), but you also request a bitInfo map that tells which atom(s) and radius set each bit.

Not a different feature type—it’s metadata for explainability/visualization.

In [4]:
def ecfp_with_bitinfo(smiles, radius=2, nBits=2048):
    mol = Chem.MolFromSmiles(smiles)
    bitInfo = {}
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=nBits, bitInfo=bitInfo)
    # bitInfo: {bit: [(atom_idx, radius_at_which_set), ...]}
    # Example: draw the first hit substructure for a bit
    bit = next(iter(bitInfo))
    atom_idx, rad = bitInfo[bit][0]
    env = Chem.FindAtomEnvironmentOfRadiusN(mol, rad, atom_idx)
    amap = {}
    submol = Chem.PathToSubmol(mol, env, atomMap=amap)
    img = Draw.MolToImage(submol, size=(250, 250))
    return fp, bitInfo, img

In [5]:
train_set=pd.read_csv('final_data/final_unique_train.csv')
test_set=pd.read_csv('final_data/final_unique_test.csv')

print(train_set.shape)
print(test_set.shape)

train_smiles_list=train_set[['smiles_canon']]
test_smiles_list=test_set[['smiles_canon']]

y_train=train_set.copy()['LogS']
y_test=test_set.copy()['LogS']

(17937, 8)
(1282, 8)


In [ ]:

print(X_train.shape)
print(X_test.shape)

(17937, 2048)
(1282, 2048)


In [ ]:
import Script.utilities as utilities

dims = [100, 200, 500, 1000, 1500, 3000, 5000, 10000]

for i in range (8, 10):
      X_train = np.vstack([morgan_binary(s, radius=i) for s in train_set['smiles_canon']])
      X_test  = np.vstack([morgan_binary(s, radius=i) for s in test_set['smiles_canon']])

      xgboost_model = xgboost.XGBRegressor(learning_rate =0.01,n_estimators=2000,max_depth=8,min_child_weight=4,gamma=0,subsample=0.7,
                                           colsample_bytree=0.8,nthread=2,scale_pos_weight=1,seed=27)
      xgboost_model.fit(X_train, y_train)

      pred = xgboost_model.predict(X_test)
      # print("RMSE:", mean_squared_error(y_test, pred, squared=False),
      # "MAE:", mean_absolute_error(y_test, pred),
      # "R2:", r2_score(y_test, pred))

      rf_491=utilities.get_errors1(y_test,pred,f'morgan Count radius {i}')
      rf_491['Descriptors_Detail']=f'morgan Coun radius {i}'

      print("Radius ", i)
      print(rf_491)

radius  8
              Model_Name     MAE     MSE    RMSE      R2    Descriptors_Detail
0  morgan Count radius 8  0.7753  1.0739  1.0363  0.7425  morgan Coun radius 8
radius  9
              Model_Name     MAE    MSE    RMSE      R2    Descriptors_Detail
0  morgan Count radius 9  0.7732  1.066  1.0325  0.7444  morgan Coun radius 9


In [18]:
rf_491

,Model_Name,MAE,MSE,RMSE,R2,Descriptors_Detail
0,morgan Count,0.8177,1.1772,1.085,0.7177,morgan Coun


In [ ]:
radius  1
              Model_Name     MAE     MSE    RMSE      R2    Descriptors_Detail
0  morgan Count radius 1  0.8452  1.2335  1.1106  0.7042  morgan Coun radius 1
radius  2
              Model_Name     MAE     MSE   RMSE      R2    Descriptors_Detail
0  morgan Count radius 2  0.8177  1.1772  1.085  0.7177  morgan Coun radius 2
radius  3
              Model_Name     MAE    MSE    RMSE      R2    Descriptors_Detail
0  morgan Count radius 3  0.8034  1.134  1.0649  0.7281  morgan Coun radius 3
radius  4
              Model_Name     MAE     MSE    RMSE      R2    Descriptors_Detail
0  morgan Count radius 4  0.7877  1.1079  1.0526  0.7344  morgan Coun radius 4

radius  5
              Model_Name     MAE    MSE    RMSE      R2    Descriptors_Detail
0  morgan Count radius 5  0.7768  1.083  1.0407  0.7403  morgan Coun radius 5
radius  6
              Model_Name    MAE     MSE    RMSE      R2    Descriptors_Detail
0  morgan Count radius 6  0.773  1.0745  1.0366  0.7424  morgan Coun radius 6
radius  7
              Model_Name     MAE     MSE    RMSE      R2    Descriptors_Detail
0  morgan Count radius 7  0.7739  1.0741  1.0364  0.7425  morgan Coun radius 7
radius  8
              Model_Name     MAE     MSE    RMSE      R2    Descriptors_Detail
0  morgan Count radius 8  0.7753  1.0739  1.0363  0.7425  morgan Coun radius 8
radius  9
              Model_Name     MAE    MSE    RMSE      R2    Descriptors_Detail
0  morgan Count radius 9  0.7732  1.066  1.0325  0.7444  morgan Coun radius 9


## Different bit sizes

In [9]:
import Script.utilities as utilities

dims = [100, 200, 500, 1000, 1500, 3000, 5000, 10000]

for i in dims:
      X_train = np.vstack([morgan_binary(s, radius=6, nBits = i) for s in train_set['smiles_canon']])
      X_test  = np.vstack([morgan_binary(s, radius=6, nBits = i) for s in test_set['smiles_canon']])

      xgboost_model = xgboost.XGBRegressor(learning_rate =0.01,n_estimators=2000,max_depth=8,min_child_weight=4,gamma=0,subsample=0.7,
                                           colsample_bytree=0.8,nthread=2,scale_pos_weight=1,seed=27)
      xgboost_model.fit(X_train, y_train)

      pred = xgboost_model.predict(X_test)
      # print("RMSE:", mean_squared_error(y_test, pred, squared=False),
      # "MAE:", mean_absolute_error(y_test, pred),
      # "R2:", r2_score(y_test, pred))

      rf_491=utilities.get_errors1(y_test,pred,f'morgan Count radius {i}')
      rf_491['Descriptors_Detail']=f'morgan Coun radius {i}'

      print("Dkimension ", i)
      print(rf_491)

Dkimension  100
                Model_Name     MAE     MSE    RMSE      R2  \
0  morgan Count radius 100  0.8849  1.4299  1.1958  0.6572   

       Descriptors_Detail  
0  morgan Coun radius 100  
Dkimension  200
                Model_Name     MAE     MSE    RMSE      R2  \
0  morgan Count radius 200  0.7847  1.1447  1.0699  0.7255   

       Descriptors_Detail  
0  morgan Coun radius 200  
Dkimension  500
                Model_Name     MAE     MSE    RMSE      R2  \
0  morgan Count radius 500  0.7463  1.0328  1.0163  0.7524   

       Descriptors_Detail  
0  morgan Coun radius 500  
Dkimension  1000
                 Model_Name     MAE     MSE    RMSE      R2  \
0  morgan Count radius 1000  0.7563  1.0436  1.0216  0.7498   

        Descriptors_Detail  
0  morgan Coun radius 1000  
Dkimension  1500
                 Model_Name     MAE     MSE    RMSE      R2  \
0  morgan Count radius 1500  0.7618  1.0545  1.0269  0.7472   

        Descriptors_Detail  
0  morgan Coun radius 1500  
Dkime